# Setup

In [1]:
# Must run on new server launch
# !pip install -r requirements.txt

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not ((repo_root / "Final").exists() and (repo_root / "Sprint 3").exists()):
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repo root.")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
from datetime import datetime
import time
from typing import Any
import pandas as pd
from pprint import pprint, pp
from tqdm.auto import tqdm
from IPython.display import clear_output, display

from Final.config import default_config
from Final.paths import FINAL_ROOT
from Final.shared_utils import setup_logging, get_logger, tail_text_file

from Final.models import (
    ExperimentState,
)
from Final.experiment_controller import ExperimentController
from Final.grid_search import GridSearchController
from Final.work_unit_scheduler import WorkUnitScheduler
from Final.coordination import CoordinationManager
from Final.pipeline_runtime import execute_pipeline_section
from Final.artifact_store import (
    LocalArtifactStore,
    DriveRegistryArtifactStore,
    HybridArtifactStore,
)
from Final.gating import (
    evaluate_module_card,
    decide_module_status,
    module_cards_to_frame,
)

from Final.labeling.pipeline import LabelingPipeline, LabelingPipelineConfig
from Final.features.pipeline import FeaturePipeline, FeaturePipelineConfig, build_feature_pipeline_ops

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
cfg = default_config()

logger = setup_logging(
    name="shrub",
    log_dir=cfg.output.logs_root,
    log_filename="main_pipeline.log",
    force=True,
)

logger.info("Initialized main pipeline notebook.")
logger.info("FINAL_ROOT = %s", FINAL_ROOT)

MAIN_OUTPUT_ROOT = cfg.output.root / "main"
MAIN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MAIN_ROOT = MAIN_OUTPUT_ROOT

MAIN_MANIFEST_DIR = MAIN_OUTPUT_ROOT / "manifests"
MAIN_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS_ROOT = MAIN_ROOT / "experiments"
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

2026-04-18 09:48:07 | INFO     | shrub | Initialized main pipeline notebook.
2026-04-18 09:48:07 | INFO     | shrub | FINAL_ROOT = /home/jovyan/work/Dry-shRub/shrub/Final


In [3]:
controller = ExperimentController(
    experiment_root=cfg.output.root / "experiments",
    experiment_name="shrubwise_main",
)

state = controller.load_state()

pipelines = {}

local_store = LocalArtifactStore(
    repo_root=cfg.data.project_root,
    storage_root=cfg.output.root / "artifact_store_local",
)

USE_DRIVE = True

if USE_DRIVE:
    drive_store = DriveRegistryArtifactStore(
        repo_root=cfg.data.project_root,
        registry_path=cfg.output.root / "artifact_registry.yaml",
        drive_config_path=cfg.data.project_root / "drive_config.yaml",
        client_secrets_path=cfg.data.project_root / "client_secrets.json",
        credentials_path=cfg.data.project_root / "pydrive_credentials.json",
    )
    artifact_store = HybridArtifactStore(
        local_store=local_store,
        remote_store=drive_store,
    )
else:
    artifact_store = local_store

grid = GridSearchController(controller=controller)

# Labeling

In [4]:
labeling_cfg = LabelingPipelineConfig(
    sprint3_variant="revised",
    sprint3_variants=("original", "revised"),
    run_sprint3=True,
    max_ptx_per_site=1,
    force_rerun_sprint3=False,
    require_success_artifacts_sprint3=True,
    cleanup_ptx_after_all_variants=True,
    cleanup_stale_ptx_before_run=True,
    stale_ptx_days=2,
    use_shape_descriptors=True,
    use_temporal_confidence=True,
    use_boundary_confidence=True,
    boundary_confidence_mode="universal",
    use_transform_confidence=False,
    use_object_subspace_filter=False,
    rasterization_mode="circle",
    multires=cfg.raster.create_multires,
    site_reference_dates={},
    subspace_min_component_pixels=4,
    subspace_min_object_confidence=0.55,
    subspace_min_transform_confidence=0.50,
    subspace_min_temporal_confidence=0.40,
    subspace_max_height_m=3.5,
    force_rerun_sprint4=False,
    force_refresh_site_assets=True,
    nonfatal_qa_overlay=True,
    allow_adopt_global_outputs=False,
)

cfg.labeling_runtime.storage.enable_local_store = True
cfg.labeling_runtime.storage.enable_drive_store = True
cfg.labeling_runtime.storage.use_hybrid_store = True

cfg.labeling_runtime.storage_policy.push_large_artifacts_to_remote = True
cfg.labeling_runtime.storage_policy.prune_local_after_remote_push = True
cfg.labeling_runtime.storage_policy.verify_remote_before_prune = True

labeling_pipeline = LabelingPipeline(cfg, pipeline_config=labeling_cfg)
base_pipeline = LabelingPipeline(cfg, pipeline_config=labeling_cfg)

pipelines["labeling"] = labeling_pipeline

print("Artifact store type:", type(labeling_pipeline.artifact_store).__name__)
print("Local store enabled:", cfg.labeling_runtime.storage.enable_local_store)
print("Drive store enabled:", cfg.labeling_runtime.storage.enable_drive_store)
print("Hybrid mode:", cfg.labeling_runtime.storage.use_hybrid_store)

2026-04-17 06:12:53 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:53 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
Artifact store type: HybridArtifactStore
Local store enabled: True
Drive store enabled: True
Hybrid mode: True


In [5]:
storage_cfg = cfg.artifact_store

print("client_secrets exists:", Path(storage_cfg.drive_client_secrets_path).exists(), storage_cfg.drive_client_secrets_path)
print("drive_config exists:", Path(storage_cfg.drive_config_path).exists(), storage_cfg.drive_config_path)
print("credentials exists:", Path(storage_cfg.drive_client_secrets_path).exists(), storage_cfg.drive_client_secrets_path)
print("drive_registry exists:", Path(storage_cfg.drive_registry_path).exists(), storage_cfg.drive_registry_path)

store = labeling_pipeline.artifact_store
print("Resolved artifact store:", type(store).__name__)

if hasattr(store, "local_store"):
    print("Hybrid local store root:", store.local_store.storage_root)
if hasattr(store, "remote_store"):
    print("Hybrid remote store type:", type(store.remote_store).__name__)
elif type(store).__name__ != "LocalArtifactStore":
    print("Remote store active:", type(store).__name__)

client_secrets exists: True /home/jovyan/work/Dry-shRub/shrub/client_secrets.json
drive_config exists: True /home/jovyan/work/Dry-shRub/shrub/drive_config.yaml
credentials exists: True /home/jovyan/work/Dry-shRub/shrub/client_secrets.json
drive_registry exists: True /home/jovyan/work/Dry-shRub/shrub/Final/artifact_registry.yaml
Resolved artifact store: HybridArtifactStore
Hybrid local store root: /home/jovyan/work/Dry-shRub/shrub/Final/artifact_store_local
Hybrid remote store type: DriveRegistryArtifactStore


In [6]:
rr = labeling_pipeline.runtime_report()
pprint(rr)

for stage_name in ["sprint3", "standardize", "refine", "transfer", "rasterize"]:
    ok, elig = labeling_pipeline.stage_is_eligible(stage_name, runtime_report=rr)
    pprint((stage_name, ok, {k: v.status.value for k, v in elig.items()}))

#display(labeling_pipeline.pipeline_spec)
display(grid.pipeline_module_state_frame(labeling_pipeline))

RuntimeCapabilityReport(detected_image_key='shrubs-labels-v1',
                        detected_image_alias='pramonettivega/shrubs-labels:v1',
                        detected_conda_env='base',
                        capabilities=['runtime:features',
                                      'runtime:labeling_transfer',
                                      'runtime:modeling',
                                      'runtime:pdal',
                                      'runtime:python',
                                      'runtime:rasterio'],
                        available_executables=['python', 'pdal'],
                        available_python_modules=['numpy',
                                                  'pandas',
                                                  'rasterio',
                                                  'scipy'],
                        marker_files_found=[],
                        marker_env_matches={'JUPYTER_IMAGE_SPEC': 'pramonettivega/shrubs-labels:v1'}

,pipeline,stage_name,module_name,enabled,variant_name,params
0,labeling,sprint3,labeling.sprint3.execution,True,"('original', 'revised')","{'max_ptx_per_site': 1, 'force_rerun_sprint3':..."
1,labeling,standardize,labeling.standardize.base,True,default,{}
2,labeling,refine,labeling.refine.shape_descriptors,True,enabled,{}
3,labeling,refine,labeling.refine.temporal_confidence,True,enabled,{'site_reference_dates': {}}
4,labeling,refine,labeling.refine.object_subspace_filter,False,disabled,"{'subspace_min_object_confidence': 0.55, 'subs..."
5,labeling,transfer,labeling.transfer.base,True,default,{}
6,labeling,rasterize,labeling.rasterize.mode,True,circle,{}
7,labeling,rasterize,labeling.boundary_confidence,True,universal,{}
8,labeling,rasterize,labeling.mask_subspace_reduction,False,disabled,{'subspace_min_component_pixels': 4}
9,labeling,rasterize,labeling.multires_export,True,default,"{'multires': (1.0, 2.0, 5.0, 10.0)}"


In [7]:
labeling_space_df = grid.section_space_frame(labeling_pipeline).copy()
display(labeling_space_df)
print("Total labeling variants:", len(labeling_space_df))

,config_signature,sprint3_variants,use_temporal_confidence,boundary_confidence_mode,use_object_subspace_filter,max_ptx_per_site
0,01236fe9e16c,"original,revised",False,radial,False,1
1,0bab4176c29e,"original,revised",True,universal,False,1
2,15a87fd1fa6a,revised,True,radial,False,1
3,2a5857f983d5,revised,False,radial,True,1
4,3628fb5a1777,"original,revised",True,radial,False,1
5,5960d3bed0d8,revised,False,universal,False,1
6,69512a22f6c7,"original,revised",True,radial,True,1
7,7568adeff82a,revised,True,universal,False,1
8,801b3fe64605,"original,revised",False,universal,False,1
9,8475fdd27e30,revised,True,radial,True,1


Total labeling variants: 16


## One Run

In [22]:
TRIAL_ID = "labeling_only_trial_001"

trial_path = controller.trial_path(TRIAL_ID)
if trial_path.exists():
    trial = controller.load_trial(TRIAL_ID)
else:
    trial = controller.create_trial(trial_id=TRIAL_ID)

controller.set_section_config(
    trial,
    "labeling",
    labeling_pipeline.config_dict(),
)
controller.save_trial(trial)
#trial

PosixPath('/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/experiments/shrubwise_main/trials/labeling_only_trial_001.json')

In [23]:
labeling_result, state, runtime_stats = execute_pipeline_section(
    labeling_pipeline,
    state=state,
    artifact_store=artifact_store,
    push_remote=USE_DRIVE,
)

controller.record_section_result(
    trial,
    section_name="labeling",
    config_signature=labeling_pipeline.config_signature(),
    result=labeling_result,
)
controller.save_trial(trial)
controller.save_state(state)

labeling_result, runtime_stats

[artifact_store] Using cached Google Drive credentials.
2026-04-16 21:34:10 | INFO     | shrub.labeling.pipeline | JSON ARTIFACT MISS | key=run_manifest | rel_path=labeling/0bab4176c29e/run_manifest.json
2026-04-16 21:34:10 | INFO     | shrub.labeling.pipeline | Skipping stage=sprint3 | reason=labeling.sprint3.execution: missing=['runtime:intelimon_sprint3']
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | Using module-aware standardize cache | data=3228b0743a1a2741 | config=a368033307bb3009
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | STORAGE POLICY RECONCILE | stage=standardize | cache_dir=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/stage_cache/standardize/3228b0743a1a2741__a368033307bb3009 | n_artifacts=1
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | STORAGE POLICY RECONCILE DONE | stage=standardize | n_artifacts=1 | statuses={'objects_csv': 'reconciled_local_path'}
2026-04-16 21:34:11 | INFO     | shrub.labeling.pipeline | Using m

(PipelineRunResult(pipeline_name='labeling', success=True, status='success', raster_outputs=CanonicalRasterOutputs(labels=                  site_id                plot_id    plot_key   variant  \
 0     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 4     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 1     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 5     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 2     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 6     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 3     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009  original   
 7     calaveras-big-trees  CATCU_0009_20250615_1  CATCU_0009   revised   
 8                dl-bliss  CAAEU_0027_20250728_1  CAAEU_0027  original   
 12               dl-bliss  CAAEU_0027_20250728_1  CAAEU_0027   revised   
 9                dl-bliss  CAAEU_0027_20250728_1  CAA

In [ ]:
# stderr_path = "/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/calaveras-big-trees/sprint3/original/CATCU_0009_20250615_1/stderr.log"
# stdout_path = "/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/calaveras-big-trees/sprint3/original/CATCU_0009_20250615_1/stdout.log"

# print("================ FULL STDERR ================")
# if os.path.exists(stderr_path):
#     with open(stderr_path, 'r') as f:
#         print(f.read())
# else:
#     print("stderr.log not found!")

# print("\n================ FULL STDOUT ================")
# if os.path.exists(stdout_path):
#     with open(stdout_path, 'r') as f:
#         print(f.read())
# else:
#     print("stdout.log not found!")

In [8]:
#print("Experiment state:")
#display(state)

# print("Trial:")
# display(trial)

print("Completed trials:")
display(grid.completed_trials_frame())

Completed trials:


,trial_id,status,n_section_runs,n_work_units,total_units,complete_units,runnable_units,blocked_units,failed_units,ineligible_units,claimed_units,running_units
0,labeling_trial_0bab4176c29e,success,0,15,15.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0
1,labeling_trial_7568adeff82a,success,0,9,9.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0
2,labeling_trial_88fb2c6936cb,success,0,15,15.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0
3,labeling_trial_e104475dd343,success,0,9,9.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0


## Full Run

In [8]:
coordination = CoordinationManager(
    artifact_store=base_pipeline.artifact_store,
    root_prefix=cfg.coordination.root_prefix,
)
scheduler = WorkUnitScheduler(controller=controller, coordination=coordination)

config_space = base_pipeline.enumerate_config_space()

len(config_space), config_space[0]

(16,
 {'sprint3_variant': 'revised',
  'use_shape_descriptors': True,
  'use_temporal_confidence': False,
  'use_boundary_confidence': True,
  'use_transform_confidence': False,
  'use_object_subspace_filter': False,
  'rasterization_mode': 'circle',
  'multires': (1.0, 2.0, 5.0, 10.0),
  'force_rerun_sprint4': False,
  'force_refresh_site_assets': True,
  'nonfatal_qa_overlay': True,
  'run_sprint3': True,
  'sprint3_variants': ('revised',),
  'max_ptx_per_site': 1,
  'force_rerun_sprint3': False,
  'require_success_artifacts_sprint3': True,
  'cleanup_ptx_after_all_variants': True,
  'cleanup_stale_ptx_before_run': True,
  'stale_ptx_days': 2,
  'boundary_confidence_mode': 'radial',
  'site_reference_dates': {},
  'subspace_min_component_pixels': 4,
  'subspace_min_object_confidence': 0.55,
  'subspace_min_transform_confidence': 0.5,
  'subspace_min_temporal_confidence': 0.4,
  'subspace_max_height_m': 3.5,
  'allow_adopt_global_outputs': False})

In [9]:
trials = []
pipelines_by_trial = {}

for config_dict in config_space:
    pipeline_cfg = LabelingPipelineConfig(**config_dict)
    pipeline = LabelingPipeline(cfg, pipeline_config=pipeline_cfg)

    trial = grid.get_or_create_trial_for_config(
        pipeline=pipeline,
        config_dict=config_dict,
        trial_prefix="labeling_trial",
    )

    trials.append(trial)
    pipelines_by_trial[trial.trial_id] = {"labeling": pipeline}

2026-04-17 06:12:54 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:54 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:55 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:56 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:56 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:56 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:57 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:57 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:57 | INFO     | shrub.labeling.pipeline | Using HybridArtifactStore for labeling pipeline
2026-04-17 06:12:58 | INFO     | shru

In [10]:
preflight_cache = grid.build_preflight_cache(
    trials=trials,
    pipelines_by_trial=pipelines_by_trial,
    show_progress=True,
    log_details=True,
)

runtime_health_df = grid.trial_runtime_health_frame_from_cache(
    preflight_cache=preflight_cache,
)

dependency_health_df = grid.trial_dependency_health_frame_from_cache(
    preflight_cache=preflight_cache,
)

preflight_df = grid.grid_preflight_frame_from_cache(
    preflight_cache=preflight_cache,
)

Building preflight cache:   0%|          | 0/16 [00:00<?, ?pipeline/s]

2026-04-17 06:13:01 | INFO     | shrub.labeling.pipeline | GRID PREFLIGHT START | trial=labeling_trial_9b7b2b083481 | pipeline=labeling
2026-04-17 06:13:01 | INFO     | shrub.labeling.pipeline | GRID PREFLIGHT SNAPSHOT | pipeline=labeling | trial=labeling_trial_9b7b2b083481 | step=runtime_report | dt=0.00s
2026-04-17 06:13:02 | INFO     | shrub.labeling.pipeline | ENUM WORK UNITS START | trial=labeling_trial_9b7b2b083481 | pipeline=labeling | config=9b7b2b083481 | register_shared_requirements=False | runtime_image=shrubs-labels-v1
2026-04-17 06:13:02 | INFO     | shrub.labeling.pipeline | ENUM WORK UNITS | trial=labeling_trial_9b7b2b083481 | stage=site_assets | n_units=6
2026-04-17 06:13:03 | INFO     | shrub.labeling.pipeline | ENUM WORK UNITS | trial=labeling_trial_9b7b2b083481 | stage=sprint3 | sprint3_complete=True | sprint3_ok=False
2026-04-17 06:13:03 | INFO     | shrub.labeling.pipeline | ENUM WORK UNITS | trial=labeling_trial_9b7b2b083481 | stage=standardize | std_complete=True

In [11]:
display(runtime_health_df)
display(dependency_health_df)
display(preflight_df)

,pipeline_name,stage_name,module_key,enabled,variant_name,runtime_eligible,missing_capabilities,reason,detected_image_key,trial_id
0,labeling,sprint3,labeling.sprint3.execution,True,"('revised',)",False,[runtime:intelimon_sprint3],runtime requirement not satisfied,shrubs-labels-v1,labeling_trial_9b7b2b083481
1,labeling,standardize,labeling.standardize.base,True,default,True,[],eligible,shrubs-labels-v1,labeling_trial_9b7b2b083481
2,labeling,refine,labeling.refine.shape_descriptors,True,enabled,True,[],eligible,shrubs-labels-v1,labeling_trial_9b7b2b083481
3,labeling,refine,labeling.refine.temporal_confidence,False,disabled,True,[],eligible,shrubs-labels-v1,labeling_trial_9b7b2b083481
4,labeling,refine,labeling.refine.object_subspace_filter,False,disabled,True,[],eligible,shrubs-labels-v1,labeling_trial_9b7b2b083481
...,...,...,...,...,...,...,...,...,...,...
155,labeling,transfer,labeling.transfer.base,True,default,True,[],eligible,shrubs-labels-v1,labeling_trial_8a2bf48671e7
156,labeling,rasterize,labeling.rasterize.mode,True,circle,True,[],eligible,shrubs-labels-v1,labeling_trial_8a2bf48671e7
157,labeling,rasterize,labeling.boundary_confidence,True,universal,True,[],eligible,shrubs-labels-v1,labeling_trial_8a2bf48671e7
158,labeling,rasterize,labeling.mask_subspace_reduction,True,enabled,True,[],eligible,shrubs-labels-v1,labeling_trial_8a2bf48671e7


,trial_id,pipeline_name,config_signature,unit_id,stage_name,scope,status,runtime_eligible,dependency_ready,dependencies,dependency_reasons,priority,site_id,plot_id,source_version
0,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,labeling_trial_9b7b2b083481:labeling:site_asse...,site_assets,site,pending,True,True,[],[],5,calaveras-big-trees,None,None
1,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,labeling_trial_9b7b2b083481:labeling:site_asse...,site_assets,site,pending,True,True,[],[],5,dl-bliss,None,None
2,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,labeling_trial_9b7b2b083481:labeling:site_asse...,site_assets,site,pending,True,True,[],[],5,independence-lake,None,None
3,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,labeling_trial_9b7b2b083481:labeling:site_asse...,site_assets,site,pending,True,True,[],[],5,pacific-union-college,None,None
4,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,labeling_trial_9b7b2b083481:labeling:site_asse...,site_assets,site,pending,True,True,[],[],5,sedgwick,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,labeling_trial_8a2bf48671e7:labeling:transfer:...,transfer,plot,blocked,True,False,[site_assets:pacific-union-college],[Shared site assets missing for site=pacific-u...,100,pacific-union-college,CALNU_0039_20250810_1,sprint3_revised
284,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,labeling_trial_8a2bf48671e7:labeling:transfer:...,transfer,plot,blocked,True,False,[site_assets:sedgwick],[Shared site assets missing for site=sedgwick],100,sedgwick,CASBC_0087_20250712_1,sprint3_original
285,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,labeling_trial_8a2bf48671e7:labeling:transfer:...,transfer,plot,blocked,True,False,[site_assets:sedgwick],[Shared site assets missing for site=sedgwick],100,sedgwick,CASBC_0087_20250712_1,sprint3_revised
286,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,labeling_trial_8a2bf48671e7:labeling:transfer:...,transfer,plot,blocked,True,False,[site_assets:shaver-lake],[Shared site assets missing for site=shaver-lake],100,shaver-lake,CAFKU_0096_20240804_1,sprint3_original


,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units
0,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
1,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,standardize,True,True,complete,[],[],1,1,0,0,0,0
2,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,refine,True,True,complete,[],[],1,1,0,0,0,0
3,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
4,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,rasterize,True,True,pending,[],[],0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
76,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,shrubs-labels-v1,standardize,True,True,complete,[],[],1,1,0,0,0,0
77,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,shrubs-labels-v1,refine,True,True,complete,[],[],1,1,0,0,0,0
78,labeling_trial_8a2bf48671e7,labeling,8a2bf48671e7,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:dl-bliss, site_assets:independenc...",12,2,0,10,0,0


In [12]:
display(preflight_df[preflight_df["runtime_eligible"] == False])
display(preflight_df[preflight_df["dependency_ready"] == False])
display(preflight_df[preflight_df["status"].isin(["blocked", "failed"])])

,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units
0,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
5,labeling_trial_2a5857f983d5,labeling,2a5857f983d5,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
10,labeling_trial_5960d3bed0d8,labeling,5960d3bed0d8,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
15,labeling_trial_b8a727c60919,labeling,b8a727c60919,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
20,labeling_trial_15a87fd1fa6a,labeling,15a87fd1fa6a,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
25,labeling_trial_8475fdd27e30,labeling,8475fdd27e30,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
30,labeling_trial_7568adeff82a,labeling,7568adeff82a,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
35,labeling_trial_be110217cd7d,labeling,be110217cd7d,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
40,labeling_trial_01236fe9e16c,labeling,01236fe9e16c,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0
45,labeling_trial_a8866b4d1f65,labeling,a8866b4d1f65,shrubs-labels-v1,sprint3,False,True,complete,[runtime:intelimon_sprint3],[],1,1,0,0,0,0


,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units
3,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
8,labeling_trial_2a5857f983d5,labeling,2a5857f983d5,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
13,labeling_trial_5960d3bed0d8,labeling,5960d3bed0d8,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
18,labeling_trial_b8a727c60919,labeling,b8a727c60919,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
23,labeling_trial_15a87fd1fa6a,labeling,15a87fd1fa6a,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
28,labeling_trial_8475fdd27e30,labeling,8475fdd27e30,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
38,labeling_trial_be110217cd7d,labeling,be110217cd7d,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:dl-bliss, site_assets:independenc...",6,1,0,5,0,0
43,labeling_trial_01236fe9e16c,labeling,01236fe9e16c,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",12,0,0,12,0,0
48,labeling_trial_a8866b4d1f65,labeling,a8866b4d1f65,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",12,0,0,12,0,0
53,labeling_trial_801b3fe64605,labeling,801b3fe64605,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",12,0,0,12,0,0


,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units
3,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
8,labeling_trial_2a5857f983d5,labeling,2a5857f983d5,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
13,labeling_trial_5960d3bed0d8,labeling,5960d3bed0d8,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
18,labeling_trial_b8a727c60919,labeling,b8a727c60919,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
23,labeling_trial_15a87fd1fa6a,labeling,15a87fd1fa6a,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
28,labeling_trial_8475fdd27e30,labeling,8475fdd27e30,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",6,0,0,6,0,0
38,labeling_trial_be110217cd7d,labeling,be110217cd7d,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:dl-bliss, site_assets:independenc...",6,1,0,5,0,0
43,labeling_trial_01236fe9e16c,labeling,01236fe9e16c,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",12,0,0,12,0,0
48,labeling_trial_a8866b4d1f65,labeling,a8866b4d1f65,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",12,0,0,12,0,0
53,labeling_trial_801b3fe64605,labeling,801b3fe64605,shrubs-labels-v1,transfer,True,False,blocked,[],"[site_assets:calaveras-big-trees, site_assets:...",12,0,0,12,0,0


In [13]:
display(grid.preflight_cache_summary_frame(preflight_cache=preflight_cache))

,trial_id,pipeline_name,config_signature,runtime_image_key,n_work_units,n_stage_rows
0,labeling_trial_9b7b2b083481,labeling,9b7b2b083481,shrubs-labels-v1,15,5
1,labeling_trial_2a5857f983d5,labeling,2a5857f983d5,shrubs-labels-v1,15,5
2,labeling_trial_5960d3bed0d8,labeling,5960d3bed0d8,shrubs-labels-v1,15,5
3,labeling_trial_b8a727c60919,labeling,b8a727c60919,shrubs-labels-v1,15,5
4,labeling_trial_15a87fd1fa6a,labeling,15a87fd1fa6a,shrubs-labels-v1,15,5
5,labeling_trial_8475fdd27e30,labeling,8475fdd27e30,shrubs-labels-v1,15,5
6,labeling_trial_7568adeff82a,labeling,7568adeff82a,shrubs-labels-v1,15,5
7,labeling_trial_be110217cd7d,labeling,be110217cd7d,shrubs-labels-v1,15,5
8,labeling_trial_01236fe9e16c,labeling,01236fe9e16c,shrubs-labels-v1,21,5
9,labeling_trial_a8866b4d1f65,labeling,a8866b4d1f65,shrubs-labels-v1,21,5


In [14]:
preflight_summary = (
    preflight_df.groupby(
        ["pipeline_name", "stage_name", "runtime_eligible", "dependency_ready", "status"],
        dropna=False
    )
    .size()
    .reset_index(name="n_rows")
    .sort_values(["pipeline_name", "stage_name", "status"])
)

display(preflight_summary)

,pipeline_name,stage_name,runtime_eligible,dependency_ready,status,n_rows
0,labeling,rasterize,True,True,pending,16
1,labeling,refine,True,True,complete,16
2,labeling,sprint3,False,True,complete,16
3,labeling,standardize,True,True,complete,16
4,labeling,transfer,True,False,blocked,14
5,labeling,transfer,True,True,complete,2


In [15]:
#progress_bar = tqdm(total=len(trials), desc="Resolved trials")
trial_bar = tqdm(total=len(trials), desc="Resolved trials", position=0)
unit_bar = tqdm(total=0, desc="Completed work units", position=1)

last_completed = {"trial_id": None, "unit_id": None, "stage_name": None}
worker_history = []
iteration_idx = 0

def render_worker_dashboard(
    *,
    trials,
    scheduler,
    active_job=None,
    candidates=None,
    tail_log_path=None,
    already_refreshed: bool = False,
    header_text: str | None = None,
):
    clear_output(wait=True)

    if already_refreshed:
        refreshed_trials = list(trials)
    else:
        refreshed_trials = [controller.load_trial(t.trial_id) for t in trials]

    resolved = sum(t.status in {"success", "failed"} for t in refreshed_trials)

    total_units = 0
    complete_units = 0
    for t in refreshed_trials:
        res = t.resolution or {}
        total_units += int(res.get("total_units", 0))
        complete_units += int(res.get("complete_units", 0))

    trial_bar.total = max(len(refreshed_trials), 1)
    trial_bar.n = resolved
    trial_bar.refresh()

    unit_bar.total = max(total_units, 1)
    unit_bar.n = complete_units
    unit_bar.refresh()

    if header_text:
        print(header_text)

    snapshot_df = scheduler.scheduler_snapshot_frame(trials=refreshed_trials)
    if not snapshot_df.empty:
        display(snapshot_df.sort_values(["status", "trial_id"]).reset_index(drop=True))

    if candidates:
        cand_df = scheduler.candidate_frame(candidates=candidates, top_n=8)
        if not cand_df.empty:
            display(cand_df)

    if active_job is not None:
        trial, pipeline, unit = active_job

        latest_trial = None
        for t in refreshed_trials:
            if t.trial_id == trial.trial_id:
                latest_trial = t
                break
        if latest_trial is None:
            latest_trial = controller.load_trial(trial.trial_id)

        current_trial_df = scheduler.current_trial_frame(trial=latest_trial)
        if not current_trial_df.empty:
            display(current_trial_df)

        display(pd.DataFrame([{
            "active_trial": trial.trial_id,
            "pipeline": pipeline.pipeline_name,
            "unit_id": unit.get("unit_id"),
            "stage_name": unit.get("stage_name"),
            "scope": unit.get("scope"),
            "site_id": unit.get("site_id"),
            "plot_id": unit.get("plot_id"),
            "source_version": unit.get("source_version"),
            "priority": unit.get("priority"),
        }]))

    if tail_log_path is not None:
        print(tail_text_file(tail_log_path, n_lines=25))

    return refreshed_trials

log_path = cfg.output.logs_root / "pipeline.log"

active_trials = [controller.load_trial(t.trial_id) for t in trials]

idle_polls = 0
max_idle_polls = 20
idle_sleep_sec = 15.0

while True:
    active_trials = [controller.load_trial(t.trial_id) for t in trials]

    # Show current known state BEFORE expensive refresh/enumeration.
    render_worker_dashboard(
        trials=active_trials,
        scheduler=scheduler,
        active_job=None,
        candidates=None,
        tail_log_path=log_path,
        already_refreshed=True,
        header_text="Collecting runnable candidates...",
    )

    force_refresh = (idle_polls == 0) or (iteration_idx == 0)
    candidates = scheduler.collect_candidate_jobs(
        trials=active_trials,
        pipelines={"labeling": labeling_pipeline},
        force_refresh=force_refresh,
    )

    # Refresh trial objects after candidate collection, since collect_candidate_jobs
    # may have updated work units/resolution on disk.
    active_trials = [controller.load_trial(t.trial_id) for t in trials]

    trial_resolution_df = pd.DataFrame([
        {
            "trial_id": t.trial_id,
            "status": t.status,
            **(t.resolution or {}),
            "n_work_units_dict": len(t.work_units or {}),
        }
        for t in active_trials
    ])
    
    display(trial_resolution_df)
    print("n_candidates:", len(candidates))

    if not candidates:
        idle_polls += 1

        active_trials = render_worker_dashboard(
            trials=active_trials,
            scheduler=scheduler,
            active_job=None,
            candidates=[],
            tail_log_path=log_path,
            already_refreshed=True,
            header_text=f"No runnable jobs found. Idle poll {idle_polls}/{max_idle_polls}",
        )

        if idle_polls >= max_idle_polls:
            break

        time.sleep(idle_sleep_sec)
        continue

    idle_polls = 0
    top = candidates[0]
    trial, pipeline, unit = top["trial"], top["pipeline"], top["unit"]

    active_trials = render_worker_dashboard(
        trials=active_trials,
        scheduler=scheduler,
        active_job=(trial, pipeline, unit),
        candidates=candidates,
        tail_log_path=log_path,
        already_refreshed=True,
        header_text="Selected next runnable job",
    )

    claimed, payload = scheduler.claim_unit(trial=trial, pipeline=pipeline, unit=unit)
    if not claimed:
        continue

    scheduler.run_claimed_job(
        trial=trial,
        pipeline=pipeline,
        unit=unit,
        state=state,
    )

    controller.save_state(state)

    iteration_idx += 1
    latest_trial = controller.load_trial(trial.trial_id)
    worker_history.append(
        {
            "iteration": iteration_idx,
            "trial_id": trial.trial_id,
            "pipeline_name": pipeline.pipeline_name,
            "unit_id": unit.get("unit_id"),
            "stage_name": unit.get("stage_name"),
            "scope": unit.get("scope"),
            "site_id": unit.get("site_id"),
            "plot_id": unit.get("plot_id"),
            "source_version": unit.get("source_version"),
            "trial_status_after": latest_trial.status,
            "resolution_after": dict(latest_trial.resolution or {}),
        }
    )

trial_bar.close()
unit_bar.close()

No runnable jobs found. Idle poll 2/20


,trial_id,status,total_units,complete_units,runnable_units,blocked_units,failed_units,ineligible_units,claimed_units,running_units
0,labeling_trial_01236fe9e16c,success,21,21,0,0,0,0,0,0
1,labeling_trial_0bab4176c29e,success,21,21,0,0,0,0,0,0
2,labeling_trial_15a87fd1fa6a,success,21,21,0,0,0,0,0,0
3,labeling_trial_2a5857f983d5,success,21,21,0,0,0,0,0,0
4,labeling_trial_3628fb5a1777,success,21,21,0,0,0,0,0,0
5,labeling_trial_5960d3bed0d8,success,21,21,0,0,0,0,0,0
6,labeling_trial_69512a22f6c7,success,21,21,0,0,0,0,0,0
7,labeling_trial_7568adeff82a,success,21,21,0,0,0,0,0,0
8,labeling_trial_801b3fe64605,success,21,21,0,0,0,0,0,0
9,labeling_trial_8475fdd27e30,success,21,21,0,0,0,0,0,0


[missing file] /home/jovyan/work/Dry-shRub/shrub/Final/artifacts/logs/pipeline.log


KeyboardInterrupt: 

In [17]:
def validate_transfer_artifacts_for_trial(trial, pipeline):
    latest = controller.load_trial(trial.trial_id)
    rows = []

    for unit in (latest.work_units or {}).values():
        if unit.get("pipeline_name") != "labeling":
            continue
        if unit.get("stage_name") != "transfer":
            continue

        site_id = unit.get("site_id")
        plot_id = unit.get("plot_id")
        source_version = unit.get("source_version")
        status = unit.get("status")

        paths = pipeline.artifact_paths_for_plot(site_id, plot_id, source_version)
        required = {
            "binary_path": paths["binary_path"].exists(),
            "confidence_path": paths["confidence_path"].exists(),
            "object_id_path": paths["object_id_path"].exists(),
            "object_table_path": paths["object_table_path"].exists(),
            "qa_path": paths["qa_path"].exists(),
        }

        rows.append({
            "trial_id": latest.trial_id,
            "site_id": site_id,
            "plot_id": plot_id,
            "source_version": source_version,
            "unit_status": status,
            **required,
            "all_required_exist": all(required.values()),
        })

    return pd.DataFrame(rows)

transfer_validation_df = pd.concat(
    [
        validate_transfer_artifacts_for_trial(trial, pipelines_by_trial[trial.trial_id]["labeling"])
        for trial in trials
    ],
    ignore_index=True,
)

display(transfer_validation_df)
display(transfer_validation_df[transfer_validation_df["all_required_exist"] == False])

,trial_id,site_id,plot_id,source_version,unit_status,binary_path,confidence_path,object_id_path,object_table_path,qa_path,all_required_exist
0,labeling_trial_9b7b2b083481,calaveras-big-trees,CATCU_0009_20250615_1,sprint3_revised,complete,False,False,False,True,False,False
1,labeling_trial_9b7b2b083481,dl-bliss,CAAEU_0027_20250728_1,sprint3_revised,complete,True,True,True,True,False,False
2,labeling_trial_9b7b2b083481,independence-lake,CATNF_6118_20250820_1,sprint3_revised,complete,True,True,True,True,True,True
3,labeling_trial_9b7b2b083481,pacific-union-college,CALNU_0039_20250810_1,sprint3_revised,complete,True,True,True,True,True,True
4,labeling_trial_9b7b2b083481,sedgwick,CASBC_0087_20250712_1,sprint3_revised,complete,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
187,labeling_trial_8a2bf48671e7,pacific-union-college,CALNU_0039_20250810_1,sprint3_revised,complete,True,True,True,True,True,True
188,labeling_trial_8a2bf48671e7,sedgwick,CASBC_0087_20250712_1,sprint3_original,complete,True,True,True,True,True,True
189,labeling_trial_8a2bf48671e7,sedgwick,CASBC_0087_20250712_1,sprint3_revised,complete,True,True,True,True,True,True
190,labeling_trial_8a2bf48671e7,shaver-lake,CAFKU_0096_20240804_1,sprint3_original,complete,True,True,True,True,True,True


,trial_id,site_id,plot_id,source_version,unit_status,binary_path,confidence_path,object_id_path,object_table_path,qa_path,all_required_exist
0,labeling_trial_9b7b2b083481,calaveras-big-trees,CATCU_0009_20250615_1,sprint3_revised,complete,False,False,False,True,False,False
1,labeling_trial_9b7b2b083481,dl-bliss,CAAEU_0027_20250728_1,sprint3_revised,complete,True,True,True,True,False,False
6,labeling_trial_9b7b2b083481,calaveras-big-trees,CATCU_0009_20250615_1,sprint3_original,complete,False,False,False,True,False,False
7,labeling_trial_9b7b2b083481,dl-bliss,CAAEU_0027_20250728_1,sprint3_original,complete,True,True,True,True,False,False
12,labeling_trial_2a5857f983d5,calaveras-big-trees,CATCU_0009_20250615_1,sprint3_revised,complete,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
171,labeling_trial_0bab4176c29e,dl-bliss,CAAEU_0027_20250728_1,sprint3_revised,complete,True,True,True,True,False,False
180,labeling_trial_8a2bf48671e7,calaveras-big-trees,CATCU_0009_20250615_1,sprint3_original,complete,False,False,False,True,False,False
181,labeling_trial_8a2bf48671e7,calaveras-big-trees,CATCU_0009_20250615_1,sprint3_revised,complete,False,False,False,True,False,False
182,labeling_trial_8a2bf48671e7,dl-bliss,CAAEU_0027_20250728_1,sprint3_original,complete,True,True,True,True,False,False


# Features

In [4]:
feature_cfg = FeaturePipelineConfig(
    canonical_grid_source=cfg.features.canonical_grid_source,
    enable_naip=True,
    enable_als=True,
    enable_3dep=True,
    enable_rap=True,
    naip_families=("raw", "veg_idx", "texture", "multiscale"),
    als_families=("height_structure",),
    dep3_families=("terrain",),
    rap_families=("prior",),
    representation_mode="both",
    enable_object_aggregation=True,
    enable_representation_export=False,
    chunk_size_px=1024,
    default_halo_px=32,
    force_refresh_site_assets=False,
    force_refresh_canonical_grid=False,
    force_refresh_chunk_manifest=False,
    force_refresh_source_ready=False,
    force_refresh_family_chunks=False,
    force_refresh_stack_finalize=False,
    force_refresh_object_aggregation=False,
)

cfg.features_runtime.storage.enable_local_store = True
cfg.features_runtime.storage.enable_drive_store = True
cfg.features_runtime.storage.use_hybrid_store = True

cfg.features_runtime.storage_policy.push_large_artifacts_to_remote = True
cfg.features_runtime.storage_policy.prune_local_after_remote_push = True
cfg.features_runtime.storage_policy.verify_remote_before_prune = True

feature_pipeline = FeaturePipeline(
    cfg,
    ops=build_feature_pipeline_ops(),
    pipeline_config=feature_cfg,
)

pipelines["features"] = feature_pipeline

print("Feature artifact store type:", type(feature_pipeline.artifact_store).__name__)

2026-04-18 09:48:13 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
Feature artifact store type: HybridArtifactStore


## Preflight Checking

In [6]:
feature_config_space = feature_pipeline.enumerate_config_space()

feature_trials = []
feature_pipelines_by_trial = {}

for config_dict in feature_config_space:
    pipe_cfg = FeaturePipelineConfig(**config_dict)
    pipe = FeaturePipeline(
        cfg,
        ops=build_feature_pipeline_ops(),
        pipeline_config=pipe_cfg,
    )

    trial = grid.get_or_create_trial_for_config(
        pipeline=pipe,
        config_dict=config_dict,
        trial_prefix="features_trial",
    )

    feature_trials.append(trial)
    feature_pipelines_by_trial[trial.trial_id] = {"features": pipe}

2026-04-18 09:48:24 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:24 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:24 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:24 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:24 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:25 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:25 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:25 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:25 | INFO     | shrub.features.pipeline | Using HybridArtifactStore for features pipeline
2026-04-18 09:48:25 | INFO     | shru

In [6]:
feature_preflight_cache = grid.build_preflight_cache(
    trials=feature_trials,
    pipelines_by_trial=feature_pipelines_by_trial,
    show_progress=True,
    log_details=True,
)

feature_runtime_health_df = grid.trial_runtime_health_frame_from_cache(
    preflight_cache=feature_preflight_cache,
)

feature_dependency_health_df = grid.trial_dependency_health_frame_from_cache(
    preflight_cache=feature_preflight_cache,
)

feature_preflight_df = grid.grid_preflight_frame_from_cache(
    preflight_cache=feature_preflight_cache,
)

Building preflight cache:   0%|          | 0/32 [00:00<?, ?pipeline/s]

2026-04-18 07:52:53 | INFO     | shrub.features.pipeline | GRID PREFLIGHT START | trial=features_trial_3cf35e4e7d44 | pipeline=features
2026-04-18 07:52:53 | INFO     | shrub.features.pipeline | GRID PREFLIGHT SNAPSHOT | pipeline=features | trial=features_trial_3cf35e4e7d44 | step=runtime_report | dt=0.00s
[artifact_store] Using cached Google Drive credentials.
2026-04-18 07:53:24 | INFO     | shrub.features.pipeline | GRID PREFLIGHT SNAPSHOT | pipeline=features | trial=features_trial_3cf35e4e7d44 | step=enumerate_work_units | n_units=54 | dt=30.52s
2026-04-18 07:53:24 | INFO     | shrub.features.pipeline | GRID PREFLIGHT SNAPSHOT | pipeline=features | trial=features_trial_3cf35e4e7d44 | step=stage_health | n_stages=8 | dt=0.00s | total=30.53s
2026-04-18 07:53:24 | INFO     | shrub.features.pipeline | GRID PREFLIGHT DONE  | trial=features_trial_3cf35e4e7d44 | pipeline=features | idx=1/32 | dt=30.53s | elapsed=30.53s | est_remaining=946.33s
2026-04-18 07:53:24 | INFO     | shrub.feature

In [7]:
display(feature_runtime_health_df)
display(feature_dependency_health_df)
display(feature_preflight_df)

,pipeline_name,stage_name,module_key,enabled,variant_name,runtime_eligible,missing_capabilities,reason,detected_image_key,trial_id
0,features,site_assets,features.site_assets.base,True,default,True,[],eligible,shrubs-labels-v1,features_trial_3cf35e4e7d44
1,features,canonical_grid,features.canonical_grid.base,True,default,True,[],eligible,shrubs-labels-v1,features_trial_3cf35e4e7d44
2,features,chunk_manifest,features.chunk_manifest.base,True,default,True,[],eligible,shrubs-labels-v1,features_trial_3cf35e4e7d44
3,features,source_ready,features.source_ready.naip,True,enabled,True,[],eligible,shrubs-labels-v1,features_trial_3cf35e4e7d44
4,features,source_ready,features.source_ready.als,True,enabled,True,[],eligible,shrubs-labels-v1,features_trial_3cf35e4e7d44
...,...,...,...,...,...,...,...,...,...,...
443,features,family_chunk,features.family_chunk.3dep,False,disabled,True,[],eligible,shrubs-labels-v1,features_trial_d552c7c0a8f7
444,features,family_chunk,features.family_chunk.rap,False,disabled,True,[],eligible,shrubs-labels-v1,features_trial_d552c7c0a8f7
445,features,stack_finalize,features.stack_finalize.base,True,default,True,[],eligible,shrubs-labels-v1,features_trial_d552c7c0a8f7
446,features,object_aggregation,features.object_aggregation.base,True,enabled,True,[],eligible,shrubs-labels-v1,features_trial_d552c7c0a8f7


,trial_id,pipeline_name,config_signature,unit_id,stage_name,scope,status,runtime_eligible,dependency_ready,dependencies,dependency_reasons,priority,site_id,plot_id,source_version
0,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,features_trial_3cf35e4e7d44:features:site_asse...,site_assets,site,complete,True,True,[],[],5,calaveras-big-trees,None,None
1,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,features_trial_3cf35e4e7d44:features:site_asse...,site_assets,site,complete,True,True,[],[],5,dl-bliss,None,None
2,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,features_trial_3cf35e4e7d44:features:site_asse...,site_assets,site,complete,True,True,[],[],5,independence-lake,None,None
3,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,features_trial_3cf35e4e7d44:features:site_asse...,site_assets,site,complete,True,True,[],[],5,pacific-union-college,None,None
4,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,features_trial_3cf35e4e7d44:features:site_asse...,site_assets,site,complete,True,True,[],[],5,sedgwick,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,features_trial_d552c7c0a8f7:features:object_ag...,object_aggregation,site,blocked,True,False,[stack_finalize],[Stack registry is not ready yet.],9500,dl-bliss,None,None
1436,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,features_trial_d552c7c0a8f7:features:object_ag...,object_aggregation,site,blocked,True,False,[stack_finalize],[Stack registry is not ready yet.],9500,independence-lake,None,None
1437,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,features_trial_d552c7c0a8f7:features:object_ag...,object_aggregation,site,blocked,True,False,[stack_finalize],[Stack registry is not ready yet.],9500,pacific-union-college,None,None
1438,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,features_trial_d552c7c0a8f7:features:object_ag...,object_aggregation,site,blocked,True,False,[stack_finalize],[Stack registry is not ready yet.],9500,sedgwick,None,None


,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units
0,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,site_assets,True,True,complete,[],[],6,6,0,0,0,0
1,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,canonical_grid,True,True,pending,[],[],6,0,6,0,0,0
2,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
3,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],24,0,0,24,0,0
4,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,family_chunk,True,True,pending,[],[],0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
252,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,family_chunk,True,True,pending,[],[],0,0,0,0,0,0
253,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,stack_finalize,True,True,pending,[],[],6,0,6,0,0,0
254,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,object_aggregation,True,False,blocked,[],[stack_finalize],6,0,0,6,0,0


In [8]:
display(feature_preflight_df[feature_preflight_df["runtime_eligible"] == False])
display(feature_preflight_df[feature_preflight_df["dependency_ready"] == False])
display(feature_preflight_df[feature_preflight_df["status"].isin(["blocked", "failed"])])

,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units


,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units
2,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
3,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],24,0,0,24,0,0
10,features_trial_69203923599e,features,69203923599e,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
11,features_trial_69203923599e,features,69203923599e,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],24,0,0,24,0,0
14,features_trial_69203923599e,features,69203923599e,shrubs-labels-v1,object_aggregation,True,False,blocked,[],[stack_finalize],6,0,0,6,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242,features_trial_6a0e34c4102f,features,6a0e34c4102f,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
243,features_trial_6a0e34c4102f,features,6a0e34c4102f,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
250,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
251,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0


,trial_id,pipeline_name,config_signature,runtime_image_key,stage_name,runtime_eligible,dependency_ready,status,missing_capabilities,blocking_dependencies,n_total_units,n_complete_units,n_pending_units,n_blocked_units,n_failed_units,n_ineligible_units
2,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
3,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],24,0,0,24,0,0
10,features_trial_69203923599e,features,69203923599e,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
11,features_trial_69203923599e,features,69203923599e,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],24,0,0,24,0,0
14,features_trial_69203923599e,features,69203923599e,shrubs-labels-v1,object_aggregation,True,False,blocked,[],[stack_finalize],6,0,0,6,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242,features_trial_6a0e34c4102f,features,6a0e34c4102f,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
243,features_trial_6a0e34c4102f,features,6a0e34c4102f,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
250,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,chunk_manifest,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0
251,features_trial_d552c7c0a8f7,features,d552c7c0a8f7,shrubs-labels-v1,source_ready,True,False,blocked,[],[canonical_grid],6,0,0,6,0,0


In [9]:
display(grid.preflight_cache_summary_frame(preflight_cache=feature_preflight_cache))

,trial_id,pipeline_name,config_signature,runtime_image_key,n_work_units,n_stage_rows
0,features_trial_3cf35e4e7d44,features,3cf35e4e7d44,shrubs-labels-v1,54,8
1,features_trial_69203923599e,features,69203923599e,shrubs-labels-v1,54,8
2,features_trial_89010190f3b0,features,89010190f3b0,shrubs-labels-v1,54,8
3,features_trial_92a9be1dafec,features,92a9be1dafec,shrubs-labels-v1,54,8
4,features_trial_42e00c1c4ab6,features,42e00c1c4ab6,shrubs-labels-v1,48,8
5,features_trial_301d6cafb488,features,301d6cafb488,shrubs-labels-v1,48,8
6,features_trial_9eb845648cf2,features,9eb845648cf2,shrubs-labels-v1,48,8
7,features_trial_52e1339ec6a3,features,52e1339ec6a3,shrubs-labels-v1,48,8
8,features_trial_cba5de331d13,features,cba5de331d13,shrubs-labels-v1,48,8
9,features_trial_4aae464e5470,features,4aae464e5470,shrubs-labels-v1,48,8


In [10]:
preflight_summary = (
    feature_preflight_df.groupby(
        ["pipeline_name", "stage_name", "runtime_eligible", "dependency_ready", "status"],
        dropna=False
    )
    .size()
    .reset_index(name="n_rows")
    .sort_values(["pipeline_name", "stage_name", "status"])
)

display(preflight_summary)

,pipeline_name,stage_name,runtime_eligible,dependency_ready,status,n_rows
0,features,canonical_grid,True,True,pending,32
1,features,chunk_manifest,True,False,blocked,32
2,features,family_chunk,True,True,pending,32
3,features,object_aggregation,True,False,blocked,16
4,features,object_aggregation,True,True,pending,16
5,features,representation_export,True,True,pending,32
6,features,site_assets,True,True,complete,32
7,features,source_ready,True,False,blocked,32
8,features,stack_finalize,True,True,pending,32


In [11]:
first_id = feature_trials[0].trial_id
last_id = feature_trials[-1].trial_id

print("FIRST:", first_id)
pprint(feature_pipelines_by_trial[first_id]["features"].config_dict())

print("\nLAST:", last_id)
pprint(feature_pipelines_by_trial[last_id]["features"].config_dict())

FIRST: features_trial_3cf35e4e7d44
{'als_families': ('height_structure',),
 'canonical_grid_source': 'naip',
 'chunk_size_px': 1024,
 'default_halo_px': 32,
 'dep3_families': ('terrain',),
 'enable_3dep': True,
 'enable_als': True,
 'enable_naip': True,
 'enable_object_aggregation': False,
 'enable_rap': True,
 'enable_representation_export': False,
 'force_refresh_canonical_grid': False,
 'force_refresh_chunk_manifest': False,
 'force_refresh_family_chunks': False,
 'force_refresh_object_aggregation': False,
 'force_refresh_site_assets': False,
 'force_refresh_source_ready': False,
 'force_refresh_stack_finalize': False,
 'naip_families': ('raw', 'veg_idx', 'texture', 'multiscale'),
 'rap_families': ('prior',),
 'representation_mode': 'raster'}

LAST: features_trial_d552c7c0a8f7
{'als_families': ('height_structure',),
 'canonical_grid_source': 'naip',
 'chunk_size_px': 1024,
 'default_halo_px': 32,
 'dep3_families': ('terrain',),
 'enable_3dep': False,
 'enable_als': False,
 'enable_n

## One Trial

In [8]:
FEATURE_TRIAL_ID = feature_trials[0].trial_id

feature_trial = controller.load_trial(FEATURE_TRIAL_ID)
feature_pipeline = feature_pipelines_by_trial[FEATURE_TRIAL_ID]["features"]

controller.set_section_config(
    feature_trial,
    "features",
    feature_pipeline.config_dict(),
)
controller.save_trial(feature_trial)

PosixPath('/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/experiments/shrubwise_main/trials/features_trial_3cf35e4e7d44.json')

In [7]:
test_assets = feature_pipeline.prepare_site_assets_for_pipeline(
    "calaveras-big-trees",
    force_refresh=False,
)
print(list(test_assets.source_assets.keys()))
print(test_assets.notes)

[artifact_store] Using cached Google Drive credentials.
2026-04-18 09:16:21 | INFO     | shrub.features.assets | Built and persisted source inventory | site=calaveras-big-trees
2026-04-18 09:16:21 | INFO     | shrub.features.source_naip | Using cached NAIP | site=calaveras-big-trees | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/notebook_cache/site_assets/calaveras-big-trees/naip/calaveras_big_trees.tif
2026-04-18 09:16:21 | INFO     | shrub.features.source_als | Using cached ALS metadata (local site cache) | site=calaveras-big-trees | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/notebook_cache/site_assets/calaveras-big-trees/als_metadata/als_metadata.json
2026-04-18 09:16:21 | INFO     | shrub.features.source_3dep | Using cached 3DEP | site=calaveras-big-trees | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/notebook_cache/site_assets/calaveras-big-trees/3dep/calaveras-big-trees_3dep_ee_10m.tif
2026-04-18 09:16:21 | INFO     | shr

In [8]:
feature_result, state, feature_runtime_stats = execute_pipeline_section(
    feature_pipeline,
    state=state,
    artifact_store=artifact_store,
    push_remote=USE_DRIVE,
)

controller.record_section_result(
    feature_trial,
    section_name="features",
    config_signature=feature_pipeline.config_signature(),
    result=feature_result,
)
controller.save_trial(feature_trial)
controller.save_state(state)

feature_result, feature_runtime_stats

2026-04-18 09:17:13 | INFO     | shrub.features.assets | Built and persisted source inventory | site=sedgwick
2026-04-18 09:17:13 | INFO     | shrub.features.source_naip | Using cached NAIP | site=sedgwick | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/notebook_cache/site_assets/sedgwick/naip/sedgwick.tif
2026-04-18 09:17:13 | INFO     | shrub.features.source_als | Using cached ALS metadata (local site cache) | site=sedgwick | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/notebook_cache/site_assets/sedgwick/als_metadata/als_metadata.json
2026-04-18 09:17:14 | INFO     | shrub.features.source_3dep | Using cached 3DEP | site=sedgwick | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/notebook_cache/site_assets/sedgwick/3dep/sedgwick_3dep_ee_10m.tif
2026-04-18 09:17:14 | INFO     | shrub.features.source_rap | Using cached RAP | site=sedgwick | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/notebook_cache/site_assets/sedg

(PipelineRunResult(pipeline_name='features', success=False, status='partial', raster_outputs=CanonicalRasterOutputs(labels=None, features=None, predictions=None, qa_overlays=None), object_outputs=CanonicalObjectOutputs(objects=None, predicted_objects=None, source_provenance=None, quality_flags=None), qa_outputs={'executed_units': ['adhoc_features:features:chunk_manifest:sedgwick', 'adhoc_features:features:chunk_manifest:shaver-lake', 'adhoc_features:features:source_ready:calaveras-big-trees:als', 'adhoc_features:features:source_ready:dl-bliss:als', 'adhoc_features:features:source_ready:independence-lake:als', 'adhoc_features:features:source_ready:pacific-union-college:als', 'adhoc_features:features:source_ready:sedgwick:als', 'adhoc_features:features:source_ready:shaver-lake:als', 'adhoc_features:features:source_ready:calaveras-big-trees:3dep', 'adhoc_features:features:source_ready:dl-bliss:3dep', 'adhoc_features:features:source_ready:independence-lake:3dep', 'adhoc_features:features:s

In [12]:
from pathlib import Path
import json

bad_trials = []

for p in sorted(controller.trials_dir.glob("*.json")):
    try:
        txt = p.read_text(encoding="utf-8")
        if not txt.strip():
            raise ValueError("empty file")
        json.loads(txt)
    except Exception as e:
        bad_trials.append((p.name, str(e), p.stat().st_size))

bad_trials

[('modeling_only_trial_002.json', 'empty file', 0),
 ('modeling_only_trial_005.json', 'empty file', 0),
 ('modeling_only_trial_006.json', 'empty file', 0),
 ('modeling_only_trial_00{i}.json', 'empty file', 0)]

In [ ]:
print("Completed trials:")
display(grid.completed_trials_frame())

In [11]:
updated_trial = controller.load_trial(FEATURE_TRIAL_ID)
display(pd.DataFrame([{
    "trial_id": updated_trial.trial_id,
    "status": updated_trial.status,
    **(updated_trial.resolution or {}),
    "n_work_units": len(updated_trial.work_units or {}),
}]))

,trial_id,status,total_units,complete_units,runnable_units,blocked_units,failed_units,ineligible_units,claimed_units,running_units,n_work_units
0,features_trial_3cf35e4e7d44,in_progress,0,0,0,0,0,0,0,0,0


## Full Run

In [9]:
feature_base_pipeline = feature_pipelines_by_trial[FEATURE_TRIAL_ID]["features"]

feature_coordination = CoordinationManager(
    artifact_store=feature_base_pipeline.artifact_store,
    root_prefix=cfg.coordination.root_prefix,
)

feature_scheduler = WorkUnitScheduler(
    controller=controller,
    coordination=feature_coordination,
)

In [10]:
for trial in feature_trials[:3]:
    print(trial.trial_id)
    print(controller.load_trial(trial.trial_id).section_configs.get("features"))
    print()

features_trial_3cf35e4e7d44
{'canonical_grid_source': 'naip', 'enable_naip': True, 'enable_als': True, 'enable_3dep': True, 'enable_rap': True, 'naip_families': ['raw', 'veg_idx', 'texture', 'multiscale'], 'als_families': ['height_structure'], 'dep3_families': ['terrain'], 'rap_families': ['prior'], 'representation_mode': 'raster', 'enable_object_aggregation': False, 'enable_representation_export': False, 'chunk_size_px': 1024, 'default_halo_px': 32, 'force_refresh_site_assets': False, 'force_refresh_canonical_grid': False, 'force_refresh_chunk_manifest': False, 'force_refresh_source_ready': False, 'force_refresh_family_chunks': False, 'force_refresh_stack_finalize': False, 'force_refresh_object_aggregation': False}

features_trial_69203923599e
{'canonical_grid_source': 'naip', 'enable_naip': True, 'enable_als': True, 'enable_3dep': True, 'enable_rap': True, 'naip_families': ['raw', 'veg_idx', 'texture', 'multiscale'], 'als_families': ['height_structure'], 'dep3_families': ['terrain'],

In [ ]:
# Check that candidate pipelines now match each trial config
feature_active_trials = [controller.load_trial(t.trial_id) for t in feature_trials]

candidates = feature_scheduler.collect_candidate_jobs(
    trials=feature_active_trials,
    pipelines=feature_pipelines_by_trial,
    force_refresh=True,
)

check_rows = []
for c in candidates[:10]:
    trial = c["trial"]
    pipeline = c["pipeline"]
    unit = c["unit"]
    check_rows.append({
        "trial_id": trial.trial_id,
        "pipeline_name": pipeline.pipeline_name,
        "pipeline_config_sig": pipeline.config_signature(),
        "trial_section_config": trial.section_configs.get("features"),
        "unit_id": unit.get("unit_id"),
        "stage_name": unit.get("stage_name"),
        "site_id": unit.get("site_id"),
    })

display(pd.DataFrame(check_rows))
print("n_candidates:", len(candidates))

2026-04-18 09:49:01 | INFO     | shrub.features.pipeline | SCHEDULER REFRESH RUN  | trial=features_trial_3cf35e4e7d44 | pipeline=features | force=True | old_fp=None | new_fp=072e928843e65a43 | age_sec=1776505741.8 | has_live_units=False
[artifact_store] Using cached Google Drive credentials.
[artifact_store] Updating existing Drive artifact for shared_artifacts/features.site_assets/9ea2fb6c27b8aaa1.json
[artifact_store] Updating existing Drive artifact for shared_artifacts/features.site_assets/9ea2fb6c27b8aaa1.json
[artifact_store] Updating existing Drive artifact for shared_artifacts/features.site_assets/cf4a75752a6cb15b.json
[artifact_store] Updating existing Drive artifact for shared_artifacts/features.site_assets/cf4a75752a6cb15b.json
[artifact_store] Updating existing Drive artifact for shared_artifacts/features.site_assets/e03b899b6aedcce5.json
[artifact_store] Updating existing Drive artifact for shared_artifacts/features.site_assets/e03b899b6aedcce5.json
[artifact_store] Updati

In [ ]:
feature_trial_bar = tqdm(total=len(feature_trials), desc="Resolved feature trials", position=0)
feature_unit_bar = tqdm(total=0, desc="Completed feature work units", position=1)

feature_worker_history = []
feature_iteration_idx = 0

feature_log_path = cfg.output.logs_root / "pipeline.log"

def render_feature_worker_dashboard(
    *,
    trials,
    scheduler,
    active_job=None,
    candidates=None,
    tail_log_path=None,
    already_refreshed: bool = False,
    header_text: str | None = None,
):
    clear_output(wait=True)

    if already_refreshed:
        refreshed_trials = list(trials)
    else:
        refreshed_trials = [controller.load_trial(t.trial_id) for t in trials]

    resolved = sum(t.status in {"success", "failed"} for t in refreshed_trials)

    total_units = 0
    complete_units = 0
    for t in refreshed_trials:
        res = t.resolution or {}
        total_units += int(res.get("total_units", 0))
        complete_units += int(res.get("complete_units", 0))

    feature_trial_bar.total = max(len(refreshed_trials), 1)
    feature_trial_bar.n = resolved
    feature_trial_bar.refresh()

    feature_unit_bar.total = max(total_units, 1)
    feature_unit_bar.n = complete_units
    feature_unit_bar.refresh()

    if header_text:
        print(header_text)

    snapshot_df = scheduler.scheduler_snapshot_frame(trials=refreshed_trials)
    if not snapshot_df.empty:
        display(snapshot_df.sort_values(["status", "trial_id"]).reset_index(drop=True))

    if candidates:
        cand_df = scheduler.candidate_frame(candidates=candidates, top_n=8)
        if not cand_df.empty:
            display(cand_df)

    if active_job is not None:
        trial, pipeline, unit = active_job

        latest_trial = None
        for t in refreshed_trials:
            if t.trial_id == trial.trial_id:
                latest_trial = t
                break
        if latest_trial is None:
            latest_trial = controller.load_trial(trial.trial_id)

        current_trial_df = scheduler.current_trial_frame(trial=latest_trial)
        if not current_trial_df.empty:
            display(current_trial_df)

        display(pd.DataFrame([{
            "active_trial": trial.trial_id,
            "pipeline": pipeline.pipeline_name,
            "unit_id": unit.get("unit_id"),
            "stage_name": unit.get("stage_name"),
            "scope": unit.get("scope"),
            "site_id": unit.get("site_id"),
            "plot_id": unit.get("plot_id"),
            "source_version": unit.get("source_version"),
            "priority": unit.get("priority"),
        }]))

    if tail_log_path is not None:
        print(tail_text_file(tail_log_path, n_lines=25))

    return refreshed_trials

feature_active_trials = [controller.load_trial(t.trial_id) for t in feature_trials]

feature_idle_polls = 0
feature_max_idle_polls = 20
feature_idle_sleep_sec = 15.0

while True:
    feature_active_trials = [controller.load_trial(t.trial_id) for t in feature_trials]

    render_feature_worker_dashboard(
        trials=feature_active_trials,
        scheduler=feature_scheduler,
        active_job=None,
        candidates=None,
        tail_log_path=feature_log_path,
        already_refreshed=True,
        header_text="Collecting runnable FE candidates...",
    )

    force_refresh = (feature_idle_polls == 0) or (feature_iteration_idx == 0)

    candidates = feature_scheduler.collect_candidate_jobs(
        trials=feature_active_trials,
        pipelines=feature_pipelines_by_trial,
        force_refresh=force_refresh,
    )

    feature_active_trials = [controller.load_trial(t.trial_id) for t in feature_trials]

    feature_trial_resolution_df = pd.DataFrame([
        {
            "trial_id": t.trial_id,
            "status": t.status,
            **(t.resolution or {}),
            "n_work_units_dict": len(t.work_units or {}),
        }
        for t in feature_active_trials
    ])

    display(feature_trial_resolution_df)
    print("n_candidates:", len(candidates))

    if not candidates:
        feature_idle_polls += 1

        feature_active_trials = render_feature_worker_dashboard(
            trials=feature_active_trials,
            scheduler=feature_scheduler,
            active_job=None,
            candidates=[],
            tail_log_path=feature_log_path,
            already_refreshed=True,
            header_text=f"No runnable FE jobs found. Idle poll {feature_idle_polls}/{feature_max_idle_polls}",
        )

        if feature_idle_polls >= feature_max_idle_polls:
            break

        time.sleep(feature_idle_sleep_sec)
        continue

    feature_idle_polls = 0
    top = candidates[0]
    trial, pipeline, unit = top["trial"], top["pipeline"], top["unit"]

    feature_active_trials = render_feature_worker_dashboard(
        trials=feature_active_trials,
        scheduler=feature_scheduler,
        active_job=(trial, pipeline, unit),
        candidates=candidates,
        tail_log_path=feature_log_path,
        already_refreshed=True,
        header_text="Selected next runnable FE job",
    )

    claimed, payload = feature_scheduler.claim_unit(
        trial=trial,
        pipeline=pipeline,
        unit=unit,
    )
    if not claimed:
        continue

    feature_scheduler.run_claimed_job(
        trial=trial,
        pipeline=pipeline,
        unit=unit,
        state=state,
    )

    controller.save_state(state)

    feature_iteration_idx += 1
    latest_trial = controller.load_trial(trial.trial_id)

    feature_worker_history.append(
        {
            "iteration": feature_iteration_idx,
            "trial_id": trial.trial_id,
            "pipeline_name": pipeline.pipeline_name,
            "unit_id": unit.get("unit_id"),
            "stage_name": unit.get("stage_name"),
            "scope": unit.get("scope"),
            "site_id": unit.get("site_id"),
            "plot_id": unit.get("plot_id"),
            "source_version": unit.get("source_version"),
            "trial_status_after": latest_trial.status,
            "resolution_after": dict(latest_trial.resolution or {}),
        }
    )

feature_trial_bar.close()
feature_unit_bar.close()

# Modeling

In [5]:
def start_model_trial(trial_name, model_name, notes, verbose=False):
    modeling_cfg = {
        "enabled": True,
        "model_family": model_name,
        "test_size": 0.2,
        "random_state": 42,
        "notes": notes,
    }

    if verbose:
        print(modeling_cfg)
    
    TRIAL_ID = trial_name
    
    trial_path = controller.trial_path(TRIAL_ID)
    if trial_path.exists():
        trial = controller.load_trial(TRIAL_ID)
    else:
        trial = controller.create_trial(trial_id=TRIAL_ID)
    
    controller.set_section_config(
        trial,
        "modeling",
        modeling_cfg,
    )
    controller.save_trial(trial)

    return trial

## log reg

In [6]:
from Final.modeling.pipeline import (
    train_log_reg, log_reg_predict, log_reg_accuracy,
    SimpleCNN, model_simple_cnn, create_cnn_wrapper,
    UNet, setup_unet_training, train_unet_model, evaluate_unet_predictions
)
from Final.modeling.pipeline import feature_permutation_pipeline, mrmr_pipeline, extract_features_logreg
from sklearn.model_selection import train_test_split
import time
import numpy as np

trial = start_model_trial("modeling_only_trial_001", "logistic_regression", "modeling pipeline supporting logistic regression")

### PLACE HOLDERS. REPLACE WITH ACTUAL DATA. 
X = np.ones((100, 5, 32, 32))  
y = np.random.randint(0, 2, (100, 1, 32, 32))

batch_size, n_channels, H, W = X.shape

# Reshape for per-pixel training: (batch*H*W, channels)
X_flat = X.reshape(batch_size * H * W, n_channels)
y_flat = y.reshape(batch_size * H * W)

X_train, X_test, y_train, y_test = train_test_split(X_flat, y_flat, test_size=trial.section_configs["modeling"]["test_size"])

# LOG REG
info = extract_features_logreg(X_train, y_train, n_mrmr_features=X.shape[1])
# model = train_log_reg(X_train, y_train)
# preds = log_reg_predict(model, X_test)
# print(preds.min(), preds.max(), preds.shape)
# acc = log_reg_accuracy(model, X_test, y_test)

trial.modeling_result = {
    "success": True,
    "status": "success",
    "metrics": info,
    "qa_outputs": {"placeholder": True},
    "notes": ["n/a"],
}

state.section_status["modeling"] = "placeholder_not_run"
state.qa_outputs["modeling"] = trial.modeling_result["qa_outputs"]

controller.save_trial(trial)
trial.modeling_result

FEATURE EXTRACTION WITH WEIGHT ZEROING
Working with 5 features

--------------------------------------------------------------------------------
STEP 1: Computing Feature Permutation Importance
--------------------------------------------------------------------------------
Computing feature permutation importance...


Feature Permutation attribution: 100%|██████████| 6/6 [00:00<00:00, 818.48it/s]


✓ Found 5 features with non-zero importance

--------------------------------------------------------------------------------
STEP 2: Running MRMR Feature Selection
--------------------------------------------------------------------------------
Computing MRMR feature selection (selecting 5 features from 5)...


0it [00:00, ?it/s]

✓ Selected 0 features via MRMR

--------------------------------------------------------------------------------
STEP 3: Combining Methods via Consensus
--------------------------------------------------------------------------------


⚠ WARNING: No consensus features selected!
  Falling back to all features with non-zero importance...
✓ Consensus selected 5 features (0.00% reduction)

--------------------------------------------------------------------------------
STEP 4: Training Logistic Regression on ALL Features
--------------------------------------------------------------------------------
✓ Model trained on all 5 features
  - Train accuracy: 0.5008

--------------------------------------------------------------------------------
STEP 5: Zeroing Out Non-Selected Feature Weights
--------------------------------------------------------------------------------
✓ Zeroed out coefficients for 0 non-selected features

PIPELINE COMPLETE

Summary:
  - Original shape: (81920, 5)
  - Origi

{'success': True,
 'status': 'success',
 'metrics': {'model': LogisticRegression(max_iter=250),
  'selected_feature_indices': [0, 1, 2, 3, 4],
  'non_selected_feature_indices': [],
  'n_original_features': 5,
  'n_selected_features': 5,
  'reduction_percentage': 0.0,
  'train_metrics': {'overall': 0.50081787109375,
   'f1': 0.0,
   'recall': 0.0,
   'precision': 0.0},
  'consensus_df':      feature  importance_score
  0  feature_0               0.0
  1  feature_1               0.0
  2  feature_2               0.0
  3  feature_3               0.0
  4  feature_4               0.0,
  'X_shape': (81920, 5),
  'permutation_features': ['feature_0',
   'feature_1',
   'feature_2',
   'feature_3',
   'feature_4'],
  'mrmr_features': []},
 'qa_outputs': {'placeholder': True},
 'notes': ['n/a']}

## simple cnn

In [7]:
from Final.modeling.pipeline import extract_features_cnn
TRIAL_ID = "modeling_only_trial_003"
trial = start_model_trial(TRIAL_ID, "simple cnn", "modeling pipeline supporting cnns")

### PLACE HOLDERS. REPLACE WITH ACTUAL DATA. 
# ***NOTE*** data should be chunked to 32 x 32. 
# as the pixel count increases, the error for shrub count in post processing increases tremendously 
X = np.ones((100,4,32,32)) 
y = np.random.randint(0, 2, (100,32,32)) # don't make 


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=trial.section_configs["modeling"]["test_size"])

results = extract_features_cnn(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    in_channels=X.shape[1], # number of bands. bands as a whole are considered as feature, rather than pixel-wise elements of a band
    num_classes=2,
    epochs=10
)
# model = SimpleCNN
# wrapper = create_cnn_wrapper(model, num_classes=2, in_channels = X.shape[1])
# train_preds, test_preds, train_metrics, test_metrics = wrapper(X_train, y_train, X_test, y_test)

trial.modeling_result = {
    "success": True,
    "status": "success",
    "metrics": {"train_metrics": results["train_metrics"], "test_metrics": results["test_metrics"]},
    "qa_outputs": {"placeholder": True},
    "notes": ["n/a"],
}

state.section_status["modeling"] = "success"
state.qa_outputs["modeling"] = trial.modeling_result["qa_outputs"]

controller.save_trial(trial)
trial.modeling_result

CNN PER-PIXEL FEATURE EXTRACTION AND TRAINING

--------------------------------------------------------------------------------
STEP 1: Initializing Model
--------------------------------------------------------------------------------
Input shape: (80, 4, 32, 32)
Target shape: (80, 32, 32)
  - Spatial dimensions: 32 × 32
  - Channels: 4
  - Pixels (features per sample): 1024
✓ Model initialized on device: cpu

--------------------------------------------------------------------------------
STEP 2: Preparing Data
--------------------------------------------------------------------------------
✓ Training data: torch.Size([80, 4, 32, 32]), torch.Size([80, 32, 32])
✓ Test data: torch.Size([20, 4, 32, 32]), torch.Size([20, 32, 32])

--------------------------------------------------------------------------------
STEP 3: Training CNN
--------------------------------------------------------------------------------
  Epoch 2/10, Loss: 0.6958
  Epoch 4/10, Loss: 0.6931
  Epoch 6/10, Loss: 0.69

{'success': True,
 'status': 'success',
 'metrics': {'train_metrics': {'accuracy': 0.4993896484375,
   'shape': (80, 32, 32)},
  'test_metrics': {'accuracy': 0.505517578125, 'shape': (20, 32, 32)}},
 'qa_outputs': {'placeholder': True},
 'notes': ['n/a']}

## unet

In [8]:
from Final.modeling.pipeline import create_dataset, calculate_unet_metrics
from Final.modeling.pipeline import UNet, extract_features_unet
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
import torch

TRIAL_ID = "modeling_only_trial_007"
trial = start_model_trial(TRIAL_ID, "unet", "modeling pipeline supporting unet")

### PLACE HOLDERS. REPLACE WITH ACTUAL DATA. 
X = np.ones((100,6,32,32)) 
y = np.random.randint(0, 2, (100,1,32,32))

dataset, train_loader = create_dataset(X,y)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False)

X = np.random.randn(100, 3, 32, 32)  # (batch, channels, H, W)
y = np.random.randint(0, 2, (100, 32, 32))  # (batch, H, W) - binary mask

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train UNet with per-pixel segmentation
results = extract_features_unet(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    in_channels=X.shape[1],
    out_channels=1,
    epochs=20,
    init_features=32
)


trial.modeling_result = {
    "success": True,
    "status": "success",
    "metrics":  results,
    "qa_outputs": {"placeholder": True},
    "notes": ["n/a"],
}

state.section_status["modeling"] = "success"
state.qa_outputs["modeling"] = trial.modeling_result["qa_outputs"]

controller.save_trial(trial)
trial.modeling_result

UNET PER-PIXEL SEGMENTATION WITH FEATURE EXTRACTION

--------------------------------------------------------------------------------
STEP 1: Initializing UNet Model
--------------------------------------------------------------------------------
Input shape: (80, 3, 32, 32)
Target shape: (80, 32, 32)
  - Spatial dimensions: 32 × 32
  - Input channels: 3
  - Output channels: 1
  - Pixels (features per sample): 1024
✓ UNet model initialized on device: cpu

--------------------------------------------------------------------------------
STEP 2: Preparing Data and DataLoaders
--------------------------------------------------------------------------------
✓ Training data: torch.Size([80, 3, 32, 32]), torch.Size([80, 1, 32, 32])
✓ Test data: torch.Size([20, 3, 32, 32]), torch.Size([20, 1, 32, 32])
✓ Batch size: 32

--------------------------------------------------------------------------------
STEP 3: Setting Up Loss, Optimizer, and Scheduler
----------------------------------------------

{'success': True,
 'status': 'success',
 'metrics': {'model': UNet(
    (encoder1): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (encoder2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inpla

# Post-processing

In [9]:
train_preds = trial.modeling_result["metrics"]["train_preds"]
test_preds = trial.modeling_result["metrics"]["test_preds"]
print(train_preds.shape, test_preds.shape)

(80, 32, 32) (20, 32, 32)


In [10]:
def start_postprocessing_trial(trial_name, method_name, notes, verbose=False):
    post_cfg = {
        "enabled": True,
        "method": method_name,
        "prob_threshold": 0.5,
        "apply_correction": method_name == "ensemble+correction",
        "notes": notes,
    }

    if verbose:
        print(post_cfg)
    
    TRIAL_ID = trial_name
    
    trial_path = controller.trial_path(TRIAL_ID)
    if trial_path.exists():
        trial = controller.load_trial(TRIAL_ID)
    else:
        trial = controller.create_trial(trial_id=TRIAL_ID)
    
    controller.set_section_config(
        trial,
        "post_processing",
        post_cfg,
    )
    controller.save_trial(trial)

    return trial

In [11]:
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'  # Headless mode for OpenGL
os.environ['LIBGL_ALWAYS_INDIRECT'] = '1'
import cv2

from Final.postprocessing.pipeline import (
    probabilities_to_predictions,
    extract_shrub_features,
    ensemble_predict_shrub_count,
    shrub_counting_pipeline
)

# Example: Ensemble-only postprocessing
trial_pp = start_postprocessing_trial(
    "postprocessing_trial_001", 
    "ensemble_only",
    "postprocessing pipeline supporting ensemble shrub counting"
)

test_preds

pp_result = shrub_counting_pipeline(
    test_preds,
    apply_correction=False,  # Use ensemble only
    verbose=True
)

# Record results to trial
trial_pp.postprocessing_result = {
    "success": True,
    "status": "success",
    "metrics": pp_result,
    "qa_outputs": {"placeholder": True},
    "notes": ["Ensemble-only shrub counting"],
}

state.section_status["postprocessing"] = "success"
state.qa_outputs["postprocessing"] = trial_pp.postprocessing_result["qa_outputs"]

controller.save_trial(trial_pp)
print("\nPostprocessing Result:")
print(trial_pp.postprocessing_result)


[Stage 2] Extracting geometric features...
  ✓ Height: 32
  ✓ Width: 32
  ✓ Aspect Ratio: 1.000

[Stage 3] Running 5-algorithm ensemble...
  CCL                 :   177
  Watershed           :   499
  Morphological       :   4
  LoG                 :   497
  DBSCAN              :   1
  Mean: 235.6, Std: 223.5
  Ensemble (after outlier rejection): 177
  ✓ Ensemble prediction: 177 shrubs

FINAL PREDICTION: 177 shrubs (ensemble_only)


Postprocessing Result:
{'success': True, 'status': 'success', 'metrics': {'ensemble_count': 177, 'corrected_count': None, 'final_count': 177, 'features': {'height': 32, 'width': 32, 'aspect_ratio': 1.0}, 'method_used': 'ensemble_only'}, 'qa_outputs': {'placeholder': True}, 'notes': ['Ensemble-only shrub counting']}


# Finalize Trial

In [ ]:
trial.qa_summary = {
    "integrity": {
        "labeling": trial.labeling_result.get("status"),
        "features": trial.features_result.get("status"),
        "modeling": trial.modeling_result.get("status"),
        "postprocessing": trial.postprocessing_result.get("status"),
    },
    "section_level": {
        "labeling": "placeholder",
        "features": "placeholder",
        "modeling": "placeholder",
        "postprocessing": "placeholder",
    },
    "cross_section": {
        "label_to_model_feedback": "placeholder",
        "feature_to_model_feedback": "placeholder",
        "end_to_end_feedback": "placeholder",
    },
}

save_trial(EXPERIMENT_NAME, trial)
trial.qa_summary

In [ ]:
labeling_object_rows = trial.labeling_result.get("metrics", {}).get("n_object_rows", 0)
labeling_artifact_rows = trial.labeling_result.get("metrics", {}).get("n_artifact_rows", 0)

trial.score_summary = {
    "labeling_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
    "features_score_placeholder": None,
    "modeling_score_placeholder": None,
    "postprocessing_score_placeholder": None,
    "composite_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
}

save_trial(EXPERIMENT_NAME, trial)
trial.score_summary

In [ ]:
trial.status = "partial" if (
    trial.features_result.get("status") == "placeholder_not_run"
    or trial.modeling_result.get("status") == "placeholder_not_run"
    or trial.postprocessing_result.get("status") == "placeholder_not_run"
) else "completed"

save_trial(EXPERIMENT_NAME, trial)

trial

In [ ]:
trial_record_min = {
    "trial_id": trial.trial_id,
    "created_at": trial.created_at,
    "status": trial.status,
    "labeling_config": trial.labeling_config,
    "features_config": trial.features_config,
    "modeling_config": trial.modeling_config,
    "postprocessing_config": trial.postprocessing_config,
    "score_summary": trial.score_summary,
}

registry.trials = [t for t in registry.trials if t["trial_id"] != trial.trial_id]
registry.trials.append(trial_record_min)

score = trial.score_summary.get("composite_score_placeholder")
if score is not None:
    if registry.best_score is None or score > registry.best_score:
        registry.best_score = score
        registry.best_trial_id = trial.trial_id

save_registry(registry)

registry

In [ ]:
trials_df = pd.DataFrame(registry.trials)

if not trials_df.empty:
    if "score_summary" in trials_df.columns:
        trials_df["composite_score_placeholder"] = trials_df["score_summary"].apply(
            lambda x: x.get("composite_score_placeholder") if isinstance(x, dict) else None
        )

    display(
        trials_df[
            ["trial_id", "created_at", "status", "composite_score_placeholder"]
        ].sort_values("trial_id")
    )

    print("Best trial:", registry.best_trial_id)
    print("Best score:", registry.best_score)
else:
    print("No trials recorded yet.")

In [ ]:
INSPECT_TRIAL_ID = trial.trial_id  # change manually

inspect_trial = load_trial(EXPERIMENT_NAME, INSPECT_TRIAL_ID)
inspect_trial